<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-02-model-architecture/lesson-2.4-gpt-and-decoder-models/notebooks/exercises-2.4.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 2.4: GPT & Decoder Models — Practice Exercises

**Netsetos GenAI Course v2.0** | Module 2 | 8 exercises: temperature, logprobs, in-context learning, structured outputs, streaming, custom sampler, perplexity, model comparison.

---

In [ ]:
!pip install transformers -q


---
## Exercise 1 (Easy): Temperature Experiment
Generate text at temperatures 0.1, 0.5, 1.0, 1.5. At what point does it produce nonsense?

In [ ]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

model = GPT2LMHeadModel.from_pretrained('gpt2')
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
prompt = 'A fun fact about India is'
inputs = tokenizer(prompt, return_tensors='pt')

print('TEMPERATURE EXPERIMENT')
print('=' * 60)
for temp in [0.1, 0.5, 1.0, 1.5]:
    output = model.generate(
        **inputs, max_new_tokens=30, do_sample=True,
        temperature=temp, top_p=0.95,
        pad_token_id=tokenizer.eos_token_id
    )
    text = tokenizer.decode(output[0], skip_special_tokens=True)
    print(f'\nTemp={temp}: {text}')

print('\n→ 0.1=repetitive, 0.5-0.7=sweet spot, >1.2=gets weird')


---
## Exercise 2 (Easy): Logprobs Explorer
Show top-5 token probabilities. Compare factual vs creative prompts.

In [ ]:
import torch
import torch.nn.functional as F
from transformers import GPT2LMHeadModel, GPT2Tokenizer

model = GPT2LMHeadModel.from_pretrained('gpt2')
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')

prompts = [
    'The capital of India is',        # factual → high confidence
    'Once upon a time there was a',    # creative → low confidence
]

for prompt in prompts:
    inputs = tokenizer(prompt, return_tensors='pt')
    with torch.no_grad():
        logits = model(**inputs).logits[0, -1, :]
    probs = F.softmax(logits, dim=-1)
    logprobs = torch.log(probs)
    top5 = torch.topk(probs, 5)
    
    print(f"\n'{prompt}'")
    for i in range(5):
        tok = tokenizer.decode(top5.indices[i])
        p = top5.values[i].item()
        lp = logprobs[top5.indices[i]].item()
        print(f"  {i+1}. '{tok}' → prob={p:.1%}, logprob={lp:.2f}")
    conf = 'HIGH' if top5.values[0] > 0.3 else 'LOW'
    print(f'  Confidence: {conf}')

print('\nFactual prompts → high confidence (one dominant token)')
print('Creative prompts → low confidence (many plausible tokens)')


---
## Exercise 3 (Medium): In-Context Learning Explorer
Test few-shot sentiment classification via prompt examples. How many examples does GPT-2 need?

In [ ]:
from transformers import pipeline

gen = pipeline('text-generation', model='gpt2', max_new_tokens=5)

# 0-shot
p0 = "Review: 'terrible movie'\nSentiment:"
r0 = gen(p0, do_sample=False)[0]['generated_text'][len(p0):].strip().split('\n')[0]
print(f'0-shot → "{r0}"')

# 1-shot
p1 = ("Review: 'amazing film!'\nSentiment: positive\n\n"
      "Review: 'terrible movie'\nSentiment:")
r1 = gen(p1, do_sample=False)[0]['generated_text'][len(p1):].strip().split('\n')[0]
print(f'1-shot → "{r1}"')

# 3-shot
p3 = ("Review: 'amazing film!'\nSentiment: positive\n"
      "Review: 'worst ever'\nSentiment: negative\n"
      "Review: 'it was okay'\nSentiment: neutral\n\n"
      "Review: 'terrible movie'\nSentiment:")
r3 = gen(p3, do_sample=False)[0]['generated_text'][len(p3):].strip().split('\n')[0]
print(f'3-shot → "{r3}"')

print('\nMore examples → better classification.')
print('This is in-context learning — the foundation of prompt engineering (Module 3)!')


---
## Exercise 4 (Medium): Structured Output Extractor
Use Gemini with response_schema to extract structured data. Compare with free-form.

In [ ]:
# This exercise requires Gemini API key
# Install: pip install google-generativeai

# import google.generativeai as genai
# genai.configure(api_key='YOUR_KEY')
# model = genai.GenerativeModel('gemini-2.0-flash')

texts = [
    'Priya Sharma, 28, from Mumbai',
    'Rahul Verma, age 42, Bangalore resident',
    'Dr. Sunita Patel, 55 years old, lives in Hyderabad',
]

print('STRUCTURED OUTPUT EXTRACTION')
print('=' * 50)
for t in texts:
    print(f"\nInput: '{t}'")
    print(f'  With schema  → {{"name": "...", "age": N, "city": "..."}}')
    print(f'  Free-form    → may return inconsistent format')

print('\nStructured outputs use constrained decoding:')
print('  At each step, tokens violating the schema get probability 0')
print('  The model CANNOT produce invalid JSON — guaranteed by design')
print('  Essential for agents (Module 8) and production APIs')


---
## Exercise 5 (Medium): Streaming Story Generator
Generate stories with stream=True. Measure time-to-first-token vs total time.

In [ ]:
# Streaming with GPT-2 (local)
from transformers import GPT2LMHeadModel, GPT2Tokenizer, TextIteratorStreamer
from threading import Thread
import time

model = GPT2LMHeadModel.from_pretrained('gpt2')
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')

prompt = 'Once upon a time in a small village in India'
inputs = tokenizer(prompt, return_tensors='pt')

streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
gen_kwargs = dict(**inputs, max_new_tokens=60, do_sample=True,
                  temperature=0.8, top_p=0.9, streamer=streamer)

start = time.time()
thread = Thread(target=model.generate, kwargs=gen_kwargs)
thread.start()

first_token_time = None
print('Streaming: ', end='', flush=True)
for token_text in streamer:
    if first_token_time is None:
        first_token_time = time.time() - start
    print(token_text, end='', flush=True)
total_time = time.time() - start
thread.join()

print(f'\n\nTime-to-first-token: {first_token_time:.3f}s')
print(f'Total generation:    {total_time:.3f}s')
print('Streaming improves perceived speed — user sees tokens immediately!')


---
## Exercise 6 (Hard): Custom Sampler with Repetition Penalty
Implement temperature → top-k → top-p → repetition penalty in correct order.

In [ ]:
import torch
import torch.nn.functional as F
from transformers import GPT2LMHeadModel, GPT2Tokenizer

def custom_sample(logits, generated_ids, temperature=1.0, top_k=50, top_p=0.9, rep_penalty=1.2):
    # Step 0: Repetition penalty
    for token_id in set(generated_ids):
        if logits[token_id] > 0:
            logits[token_id] /= rep_penalty
        else:
            logits[token_id] *= rep_penalty
    # Step 1: Temperature
    logits = logits / temperature
    # Step 2: Top-k
    if top_k > 0:
        indices_to_remove = logits < torch.topk(logits, top_k)[0][-1]
        logits[indices_to_remove] = float('-inf')
    # Step 3: Top-p
    if top_p < 1.0:
        sorted_logits, sorted_indices = torch.sort(logits, descending=True)
        cumprobs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
        remove = cumprobs > top_p
        remove[1:] = remove[:-1].clone()
        remove[0] = False
        logits[sorted_indices[remove]] = float('-inf')
    # Step 4: Sample
    probs = F.softmax(logits, dim=-1)
    return torch.multinomial(probs, 1).item()

model = GPT2LMHeadModel.from_pretrained('gpt2')
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')

prompt = 'The future of AI is'
ids = tokenizer(prompt, return_tensors='pt').input_ids[0].tolist()

for _ in range(40):
    with torch.no_grad():
        logits = model(torch.tensor([ids])).logits[0, -1, :].clone()
    next_tok = custom_sample(logits, ids, temperature=0.8, top_k=50, top_p=0.9, rep_penalty=1.2)
    ids.append(next_tok)

print('Custom sampler output:')
print(tokenizer.decode(ids))
print('\nOrder: rep_penalty → temperature → top-k → top-p → sample')


---
## Exercise 7 (Hard): Perplexity Calculator
Compute GPT-2 perplexity on 3 domains. Which has lowest? What does that reveal?

In [ ]:
import torch, math
from transformers import GPT2LMHeadModel, GPT2Tokenizer

model = GPT2LMHeadModel.from_pretrained('gpt2')
model.eval()
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')

texts = {
    'Wikipedia': 'The Taj Mahal is an ivory-white marble mausoleum on the right bank of the river Yamuna in Agra, India.',
    'Reddit': 'lol this is so random haha anyone else think AI is gonna take over xD honestly idk man',
    'Python': 'def fibonacci(n):\n    if n <= 1: return n\n    return fibonacci(n-1) + fibonacci(n-2)',
}

print('PERPLEXITY COMPARISON')
print('=' * 50)
for name, text in texts.items():
    ids = tokenizer(text, return_tensors='pt').input_ids
    with torch.no_grad():
        loss = model(ids, labels=ids).loss
    ppl = math.exp(loss.item())
    conf = 'HIGH' if ppl < 40 else 'MEDIUM' if ppl < 100 else 'LOW'
    print(f'  {name:<12} ppl={ppl:>8.1f}  confidence={conf}')

print('\nPerplexity = exp(loss) — lower = model more confident')
print('Wikipedia: lowest (closest to training data)')
print('Perplexity is THE standard metric for evaluating language models')


---
## Exercise 8 (Challenge): Model Comparison Dashboard
Same prompt to GPT-2 and Gemini. Compare speed, quality, cost.

In [ ]:
import time
from transformers import pipeline

prompt = 'Explain quantum computing in 3 sentences.'

# GPT-2 (local)
gen = pipeline('text-generation', model='gpt2')
start = time.time()
gpt2_out = gen(prompt, max_new_tokens=60, do_sample=True, temperature=0.7)[0]['generated_text']
gpt2_time = time.time() - start

print('MODEL COMPARISON DASHBOARD')
print('=' * 60)
print(f'\nGPT-2 (1.5B, local):')
print(f'  Time: {gpt2_time:.3f}s')
print(f'  Cost: $0 (runs locally)')
print(f'  Output: {gpt2_out[:150]}...')

print(f'\nGemini Flash (API — requires key):')
print(f'  Time: ~0.8-1.5s (network latency)')
print(f'  Cost: $0.15/M input tokens')
print(f'  Output: [Much better quality than GPT-2]')

print(f'\nLLaMA 3 8B (via Ollama — requires local install):')
print(f'  Time: ~2-5s (depends on GPU)')
print(f'  Cost: $0 (runs locally)')
print(f'  Output: [Near Gemini quality]')

print('DECISION: Prototype with Gemini Flash ($0.15/M).')
print('Scale with LLaMA 3 (free, private). GPT-2 is for learning only.')


---
**8 exercises complete.** 12 GPT concepts mastered: CLM, autoregressive loop, temperature, top-k/p, GPT-2, Gemini API, GPT family, scaling laws, in-context learning, logprobs, training pipeline, production controls.